# fed-llm-ids — Federated LoRA fine-tuning for NF-ToN-IoT

Trains BERT / T5 / Llama / Qwen / Gemma as federated clients with LoRA, on IID and
non-IID partitions, and compares four ways of aggregating the LoRA updates.

**Before running, set up the notebook:**

| Setting | Value |
|---|---|
| Accelerator | `GPU T4 x2` |
| Internet | `On` (needed for the Hub and Comet) |
| Input | attach the NF-ToN-IoT dataset |
| Add-ons → Secrets | `HF_TOKEN` with **write** access |

**Accept the gated licences first**, on the account that owns `HF_TOKEN`:
[Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) and
[gemma-4-E2B](https://huggingface.co/google/gemma-4-E2B). Without this the
download fails with `OSError: You are trying to access a gated repo`.

One model per session. Change `MODEL` in the *Session* cell and run top to bottom.

| # | MODEL | approx. time |
|---|---|---|
| 1 | `bert` | ~20 min |
| 2 | `t5` | ~45 min |
| 3 | `llama` | ~2.5 h |
| 4 | `qwen` | ~5 h |
| 5 | `gemma` | ~5 h |

Gemma and Llama are gated — accept their licences on the Hub first, or the
download returns 403.

## 1. Install

Kaggle ships torchao 0.10, but peft's LoRA dispatcher *raises* on any version
below 0.16 rather than skipping it — even though this pipeline never uses torchao
quantisation. Removing it makes peft's availability check return False cleanly,
which is what we want. Without this, `get_peft_model` fails for every model.

In [ ]:
!pip install -q -U transformers peft accelerate comet_ml
!pip uninstall -q -y torchao

import peft, transformers, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__)
print('GPUs:', torch.cuda.device_count())

## 2. Code and dataset

Pulls the pipeline from the `llm` branch. Re-run this cell after pushing changes
to pick them up — delete `/kaggle/working/repo` first, or the clone is skipped.

In [ ]:
import glob, os

BRANCH = 'llm'
REPO = '/kaggle/working/repo'

if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/asfi50/Fed_GNN.git {REPO}
os.chdir(REPO)
!git log -1 --oneline

candidates = glob.glob('/kaggle/input/**/*.csv', recursive=True)
CSV = next(c for c in candidates
           if 'NF-ToN-IoT' in c and 'common' not in c and 'easy' not in c and 'uncommon' not in c)

DATA = '/kaggle/working/dataset'
RESULTS = '/kaggle/working/results'

print('repo    :', os.getcwd())
print('dataset :', CSV)

## 3. Hugging Face token

Trained adapters upload automatically as `fed-ids-<model-full-name>`.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
print('HF_TOKEN loaded:', bool(os.environ.get('HF_TOKEN')))

## 4. Build the client shards

Rebuilt every session on purpose. The seed is fixed, so the partition is
byte-identical each time and nothing has to be carried between sessions.

Holds out a shared val/test set first, then splits the rest five ways twice —
once IID, once Dirichlet(0.5) non-IID. The two tables printed below are the
attack distribution per client for each variant.

In [ ]:
!python llm/split_dataset.py --input_file {CSV} --output_dir {DATA} --num_clients 5 --alpha 0.5

## 5. Session — pick the model

**This is the only cell to change between sessions.**

The first check confirms all five model ids still resolve on the Hub (seconds, no
weights). The second runs this session's model through the real pipeline: LoRA
attach, a tokenised batch, forward, backward, state round-trip, aggregation.

In [ ]:
MODEL = 'bert'      # bert -> t5 -> llama -> qwen -> gemma

!python llm/check_models.py
!python llm/check_models.py --forward --verbose --models {MODEL}

## 6. Run both splits, one per T4

Kaggle charges session wall-clock rather than GPU count, so running IID and
non-IID together gets two experiments per quota hour.

Launched with `subprocess` rather than `!cmd &` — each `!` line is its own shell,
so a following `!wait` would have nothing to wait for. Progress prints every
5 minutes; full logs are in `/kaggle/working/`.

In [ ]:
import subprocess, time

procs = []
for gpu, split in [(0, 'iid'), (1, 'non_iid')]:
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
    log = open(f'/kaggle/working/{MODEL}_{split}.log', 'w')
    procs.append(subprocess.Popen([
        'python', 'llm/run_experiment.py',
        '--model', MODEL, '--split', split,
        '--data_dir', DATA, '--output_dir', RESULTS,
    ], env=env, stdout=log, stderr=subprocess.STDOUT))
    print(f'launched {MODEL}/{split} on GPU {gpu}')

while any(p.poll() is None for p in procs):
    time.sleep(300)
    for split in ['iid', 'non_iid']:
        tail = subprocess.run(['tail', '-1', f'/kaggle/working/{MODEL}_{split}.log'],
                              capture_output=True, text=True).stdout.strip()
        print(f'[{split}] {tail[:110]}')

print('exit codes:', [p.returncode for p in procs])

### If a run failed or the session timed out

Rerun the cell above with `'--resume'` added to the argument list — training picks
up from the last completed round rather than starting over. Every round is
checkpointed to the output directory.

## 7. Results so far

In [ ]:
import json, pandas as pd

rows = []
for f in sorted(glob.glob(f'{RESULTS}/*/results.json')):
    r = json.load(open(f))
    m = r['test_metrics']
    rows.append({
        'run': r['run'],
        'accuracy': round(m['accuracy'], 4),
        'balanced_acc': round(m['balanced_accuracy'], 4),
        'macro_f1': round(m['macro_f1'], 4),
        'comm_MB': round(r['communication_mb'], 1),
        'minutes': round(r['total_seconds'] / 60, 1),
    })
pd.DataFrame(rows)

## 8. Centralized baseline (optional, once per model)

Trains on the pooled data with exactly the same budget a federated run gets —
`clients x rows_per_round x rounds` rows, one pass. The gap between this and the
federated numbers is the cost of federation, and nothing else.

In [ ]:
!python llm/run_centralized.py --model {MODEL} --split iid --data_dir {DATA} --output_dir {RESULTS}

## 9. Aggregation ablation (optional, after the five models are done)

`naive` averages the LoRA A and B matrices separately, which is what most
federated LoRA work does and is mathematically wrong: `mean(B_i @ A_i)` is not
`mean(B_i) @ mean(A_i)`. The other three are different answers to that.

Worth running on the two strongest models, non-IID only, where heterogeneity
makes the aggregation choice matter most.

In [ ]:
for agg in ['delta_svd', 'ffa', 'performance']:
    print(f'=== {MODEL} / non_iid / {agg} ===')
    !python llm/run_experiment.py --model {MODEL} --split non_iid --agg {agg} --data_dir {DATA} --output_dir {RESULTS} --no_push

## Notes

- **Comet** — everything logs to project `fed-llm-ids`, tagged by model, split and
  aggregation. Parameters include the full dataset provenance, so any result can
  be traced back to the partition that produced it.
- **Adapters** — pushed as `fed-ids-<model-full-name>`. Runs of the same model
  share one repo, so the last one wins; `results.json` and Comet keep the full
  record regardless.
- **Fairness** — every model gets an identical data budget, set by the slowest
  (the 2B ones). Do not raise it for the cheap models.
- **Leakage** — the serialiser drops `Label`, `Attack`, and both IP columns.
  Attacker addresses are near-unique per class in NF-ToN-IoT, so keeping them
  would let a model memorise addresses instead of learning traffic behaviour.